# 07 Full Inference Analyzer

This notebook combines the main ideas from Steps 1 to 6 into one unified simulated analyzer.

It calculates:
- token counts
- context usage
- prefill load and estimated TTFT
- decode load and estimated decode time
- total estimated latency
- warning messages for risky scenarios

## Important Note

This is still a simulation, not real model inference.

To turn this into a real benchmark later, you would need:
- a real model loaded
- a real inference request
- streaming output
- TTFT measurement
- generated token counting
- total latency measurement
- tokens/sec measurement
- repeated runs with average, p50, and p95 reporting

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parents[0]
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.append(str(PROJECT_ROOT / "src"))

from inference_analyzer import (
    FIGURES_DIR,
    FULL_INFERENCE_CSV_PATH,
    FULL_INFERENCE_JSON_PATH,
    FULL_PROMPT_EXAMPLE_PATH,
    create_full_analysis_graphs,
    run_full_analysis,
)

print("Project root:", PROJECT_ROOT)
print("Example prompt path:", FULL_PROMPT_EXAMPLE_PATH)
print("JSON output path:", FULL_INFERENCE_JSON_PATH)
print("CSV output path:", FULL_INFERENCE_CSV_PATH)
print("Figures directory:", FIGURES_DIR)

Project root: /Users/nikhiljuluri/Desktop/LLM_Infra_Projects/LLM_Inference_cost_Latency_analyser/week2_LLM_Inference
Example prompt path: /Users/nikhiljuluri/Desktop/LLM_Infra_Projects/LLM_Inference_cost_Latency_analyser/week2_LLM_Inference/examples/full_prompt_example.txt
JSON output path: /Users/nikhiljuluri/Desktop/LLM_Infra_Projects/LLM_Inference_cost_Latency_analyser/week2_LLM_Inference/outputs/full_inference_analysis.json
CSV output path: /Users/nikhiljuluri/Desktop/LLM_Infra_Projects/LLM_Inference_cost_Latency_analyser/week2_LLM_Inference/outputs/full_inference_analysis.csv
Figures directory: /Users/nikhiljuluri/Desktop/LLM_Infra_Projects/LLM_Inference_cost_Latency_analyser/week2_LLM_Inference/outputs/figures


## 1. Direct Text Analysis

This is the simplest usage mode. We pass text directly into the analyzer for one tokenizer.

In [2]:
results_text, df_text = run_full_analysis(
    text="Explain Docker for LLM inference.",
    tokenizer_name="Qwen/Qwen2.5-1.5B-Instruct",
    context_window=8192,
    reserved_output_tokens=512,
    expected_output_tokens=512,
    tokens_per_second=35,
)
df_text

ValueError: too many values to unpack (expected 2)

## 2. File-Based Analysis

This mode reads a prompt from a text file and analyzes it with one tokenizer.

In [ ]:
results_file, df_file = run_full_analysis(
    file_path=str(FULL_PROMPT_EXAMPLE_PATH),
    tokenizer_name="Qwen/Qwen2.5-1.5B-Instruct",
    context_window=8192,
    reserved_output_tokens=512,
    expected_output_tokens=512,
    tokens_per_second=35,
)
df_file

## 3. Compare All Tokenizers

This mode runs the same prompt across the full tokenizer set so we can compare token counts, context usage, and total estimated latency.

In [ ]:
results_compare, df_compare = run_full_analysis(
    file_path=str(FULL_PROMPT_EXAMPLE_PATH),
    compare_tokenizers=True,
    context_window=8192,
    reserved_output_tokens=512,
    expected_output_tokens=512,
    tokens_per_second=35,
)
df_compare

## Saved Outputs

The analyzer writes both JSON and CSV outputs so the results are easy to inspect programmatically or in spreadsheet form.

In [ ]:
FULL_INFERENCE_JSON_PATH.exists(), FULL_INFERENCE_CSV_PATH.exists()

## Create Graphs

These graphs use the tokenizer-comparison DataFrame because the comparison view is the most informative for Step 7.

In [ ]:
create_full_analysis_graphs(df_compare, FIGURES_DIR)
sorted(path.name for path in FIGURES_DIR.glob('full_*.png')) + sorted(path.name for path in FIGURES_DIR.glob('tokenizer_latency_comparison.png')) + sorted(path.name for path in FIGURES_DIR.glob('inference_summary.png'))

## What the Graphs Show

1. **Context usage percent by tokenizer**
   This shows how efficiently each tokenizer represents the same prompt.

2. **Latency breakdown: TTFT vs decode time**
   This separates the up-front prefill cost from the output-generation cost.

3. **Total estimated latency across tokenizers**
   This shows which tokenizer leads to the smallest or largest simulated total latency for the same prompt.

4. **Inference summary**
   This combines prompt tokens, remaining tokens, and expected output tokens to summarize the overall budget picture.

## Warning Logic

The analyzer adds warnings when:
- the prompt does not fit the context window
- remaining tokens are low
- reserved output tokens are very high
- total estimated latency is high
- prompt tokens are high and prefill load is high
- expected output tokens are high and decode load is high

## Step 7 Takeaway

By this step, we can analyze one prompt end to end: tokenize it, check whether it fits the context window, estimate prefill cost, estimate decode cost, and summarize the likely latency tradeoffs in one place.